In [ ]:
import os
import json
import pandas as pd
import streamlit as st

# Titre de l'application
st.title("🌤️ Dashboard Météo - Thiès, Sénégal")
st.write(
    "Application de suivi des données météorologiques en temps réel construite"
    " dans le cadre de mon projet de Data Engineering."
)

# Fonction pour charger toutes les données nettoyées
dossier_processed = "data_meteo/processed"


@st.cache_data
def charger_donnees():
  fichiers_a_lire = []
  if os.path.exists(dossier_processed):
    for root, dirs, files in os.walk(dossier_processed):
      for file in files:
        if file.endswith(".json"):
          fichiers_a_lire.append(os.path.join(root, file))

  data_list = []
  for f in fichiers_a_lire:
    with open(f, "r", encoding="utf-8") as file_ouvert:
      data_list.append(json.load(file_ouvert))

  if data_list:
    return pd.DataFrame(data_list)
  return pd.DataFrame()


df_meteo = charger_donnees()

# Affichage du dashboard si on a des données
if not df_meteo.empty:
  st.subheader("📊 Dernières mesures enregistrées")
  st.dataframe(df_meteo)

  # Affichage de quelques métriques clés (KPIs)
  derniere_ligne = df_meteo.iloc[-1]

  col1, col2, col3 = st.columns(3)
  with col1:
    st.metric(
        label="Température",
        value=f"{derniere_ligne['temperature_celsius']} °C",
    )
  with col2:
    st.metric(
        label="Humidité", value=f"{derniere_ligne['humidite_pourcentage']} %"
    )
  with col3:
    st.metric(label="Vent", value=f"{derniere_ligne['vent_kmh']} km/h")

  # Graphique simple de l'évolution de la température
  st.subheader("📈 Évolution de la température")
  st.line_chart(df_meteo.set_index("timestamp_mesure")["temperature_celsius"])

else:
  st.warning(
      "Aucune donnée trouvée. Exécute d'abord ton script de transformation !"
  )